In [ ]:
!pip install -q osmnx networkx shapely geopandas geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving tokyo_full_graph_updated.json to tokyo_full_graph_updated.json


In [ ]:
import json

with open("tokyo_full_graph_updated.json") as f:
    data = json.load(f)

print("Nodes:", len(data["nodes"]))
print("Edges:", len(data["edges"]))

Nodes: 87091
Edges: 197677


In [ ]:
ROAD_MAP = {
    "motorway": "highway",
    "motorway_link": "highway",
    "trunk": "highway",

    "primary": "major",
    "secondary": "major",

    "tertiary": "minor",

    "residential": "local",
    "unclassified": "local",
    "service": "local",
}

SPEEDS = {
    "highway": 25,
    "major": 16,
    "minor": 11,
    "local": 7
}

for e in data["edges"]:
    rt = e.get("road_type", "unclassified")

    road_class = ROAD_MAP.get(rt, "local")
    e["road_class"] = road_class

    speed = SPEEDS[road_class]
    e["speed"] = speed

    e["travel_time"] = e["length"] / speed

In [ ]:
import networkx as nx

G = nx.MultiDiGraph()

# Add nodes
for n in data["nodes"]:
    G.add_node(
        n["id"],
        x=n["lon"],
        y=n["lat"],
        type=n.get("type", "road")
    )

# Add edges
for e in data["edges"]:
    G.add_edge(
        e["from"],
        e["to"],
        length=e["length"],
        travel_time=e["travel_time"],
        road_class=e["road_class"]
    )

In [ ]:
from shapely.geometry import LineString

for u, v, k, d in G.edges(keys=True, data=True):
    x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
    x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
    d["geometry"] = LineString([(x1, y1), (x2, y2)])

In [ ]:
import osmnx as ox

#set coordinates system as lat/lon
G.graph["crs"] = "EPSG:4326"

# Add osmid
for i, (u, v, k, d) in enumerate(G.edges(keys=True, data=True)):
    d["osmid"] = i

G = ox.convert.to_undirected(G)

G = ox.project_graph(G)

In [ ]:
count_missing = 0

for u, v, k, d in G.edges(keys=True, data=True):
    if "geometry" not in d:
        count_missing += 1

print("Edges missing geometry:", count_missing)

Edges missing geometry: 4


In [ ]:
from shapely.geometry import LineString

fixed = 0

for u, v, k, d in G.edges(keys=True, data=True):
    if "geometry" not in d:
        x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
        x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]

        d["geometry"] = LineString([(x1, y1), (x2, y2)])
        fixed += 1

print(f"Fixed {fixed} edges with missing geometry")

Fixed 4 edges with missing geometry


In [ ]:
missing = sum(1 for _,_,_,d in G.edges(keys=True, data=True) if "geometry" not in d)
print("Remaining missing:", missing)

Remaining missing: 0


In [ ]:
print(G.get_edge_data(u, v))

{0: {'length': 3967327.5502859894}}


In [ ]:
from shapely.geometry import Point, LineString

facility_nodes = [
    (n, d) for n, d in G.nodes(data=True)
    if d.get("type") in ["hospital", "fire_station"]
]

new_node_id = max(G.nodes) + 1

for node_id, node_data in facility_nodes:

    point = Point(node_data["x"], node_data["y"])

    #  nearest edge (R-tree via OSMnx)
    u, v, key = ox.distance.nearest_edges(G, point.x, point.y)

    edge_dict = G.get_edge_data(u, v)
    edge_data = edge_dict[key]

    # --- SAFE GEOMETRY ---
    x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
    x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]

    line = edge_data.get(
        "geometry",
        LineString([(x1, y1), (x2, y2)])
    )

    # project facility onto edge
    projected = line.interpolate(line.project(point))

    # --- CREATE CONNECTOR NODE ---
    G.add_node(
        new_node_id,
        x=projected.x,
        y=projected.y,
        type="connector"
    )

    # --- REMOVE ORIGINAL EDGE ---
    G.remove_edge(u, v, key)

    # --- DISTANCES ---
    p1 = Point(x1, y1)
    p2 = Point(x2, y2)

    length1 = p1.distance(projected)
    length2 = projected.distance(p2)

    attrs = edge_data.copy()

    # --- EDGE  (u → new_node) ---
    attrs1 = attrs.copy()
    attrs1["length"] = length1

    if "speed" in attrs1:
        attrs1["travel_time"] = length1 / attrs1["speed"]

    attrs1["geometry"] = LineString([(x1, y1), (projected.x, projected.y)])

    G.add_edge(u, new_node_id, **attrs1)

    # --- EDGE  (new_node → v) ---
    attrs2 = attrs.copy()
    attrs2["length"] = length2

    if "speed" in attrs2:
        attrs2["travel_time"] = length2 / attrs2["speed"]

    attrs2["geometry"] = LineString([(projected.x, projected.y), (x2, y2)])

    G.add_edge(new_node_id, v, **attrs2)

    # --- CONNECT FACILITY ---
    connector_length = point.distance(projected)

    G.add_edge(
        node_id,
        new_node_id,
        length=connector_length,
        road_type="facility_connector",
        road_class="local",
        speed=5,
        travel_time=connector_length / 5
    )

    G.add_edge(
        new_node_id,
        node_id,
        length=connector_length,
        road_type="facility_connector",
        road_class="local",
        speed=5,
        travel_time=connector_length / 5
    )

    new_node_id += 1

print("Facilities connected with full attribute preservation")

Facilities connected with full attribute preservation


In [ ]:
missing = sum(
    1 for _,_,_,d in G.edges(keys=True, data=True)
    if "road_class" not in d
)

print("Edges missing attributes:", missing)

Edges missing attributes: 55


In [ ]:
from shapely.geometry import LineString

fixed = 0

for u, v, k, d in G.edges(keys=True, data=True):

    if "geometry" not in d:
        x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
        x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
        d["geometry"] = LineString([(x1, y1), (x2, y2)])

    if "road_class" not in d:
        d["road_class"] = "local"

    if "speed" not in d:
        d["speed"] = 7

    if "length" not in d:
        x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
        x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
        d["length"] = LineString([(x1, y1), (x2, y2)]).length

    if "travel_time" not in d:
        d["travel_time"] = d["length"] / d["speed"]

    fixed += 1

print(f" Cleaned {fixed} edges")

 Cleaned 119806 edges


In [ ]:
missing = sum(
    1 for _,_,_,d in G.edges(keys=True, data=True)
    if "road_class" not in d or "travel_time" not in d
)

print("Remaining problematic edges:", missing)

Remaining problematic edges: 0


In [ ]:
components = list(nx.connected_components(G.to_undirected()))

print("Number of components:", len(components))

Number of components: 1


In [ ]:
print("Total nodes in G:", len(G.nodes))

Total nodes in G: 77576


In [ ]:
import networkx as nx
import random

source = random.choice(list(G.nodes))

hospitals = [
    n for n, d in G.nodes(data=True)
    if d.get("type") == "hospital"
]

time_lengths, time_paths = nx.multi_source_dijkstra(
    G,
    sources=hospitals,
    weight="travel_time"
)

dist_lengths, dist_paths = nx.multi_source_dijkstra(
    G,
    sources=hospitals,
    weight="length"
)

if source in time_paths and source in dist_paths:

    print("Source:", source)

    print("\n TIME-OPTIMAL ROUTE")
    print("Hospital:", time_paths[source][-1])
    print("Time (min):", time_lengths[source] / 60)
    print("Distance (m):", dist_lengths[source])
else:
    print(" Source not reachable")

Source: 344387363

 TIME-OPTIMAL ROUTE
Hospital: 344387363
Time (min): 39.42241662353244
Distance (m): 32741.15212631961


In [ ]:
import geopandas as gpd

bridges_gdf = gpd.read_file("tokyo_bridges.geojson")

print("Bridges:", len(bridges_gdf))

Bridges: 2429


/usr/local/lib/python3.12/dist-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


In [ ]:
import osmnx as ox

for _, row in bridges_gdf.iterrows():
    geom = row.geometry

    if geom is None:
        continue

    # use centroid of bridge
    point = geom.centroid

    # find nearest node in graph
    node = ox.distance.nearest_nodes(G, point.x, point.y)

    # mark node as bridge
    G.nodes[node]["is_bridge"] = True
    G.nodes[node]["fragility"] = 0.7
    G.nodes[node]["capacity"] = 100

In [ ]:
import json

output = {
    "nodes": [],
    "edges": []
}

# --- NODES ---
for n, d in G.nodes(data=True):
    output["nodes"].append({
        "id": int(n),
        "x": float(d.get("x", 0)),
        "y": float(d.get("y", 0)),
        "type": d.get("type", "road"),
        "is_bridge": d.get("is_bridge", False),
        "fragility": d.get("fragility", None),
        "capacity": d.get("capacity", None)
    })

# --- EDGES ---
for u, v, k, d in G.edges(keys=True, data=True):
    output["edges"].append({
        "from": int(u),
        "to": int(v),
        "length": float(d.get("length", 0)),
        "travel_time": float(d.get("travel_time", 0)),
        "road_class": d.get("road_class", "local"),
        "road_type": d.get("road_type", "road")
    })

# save file
with open("final_graph.json", "w") as f:
    json.dump(output, f)

print("Graph saved as final_graph.json")

Graph saved as final_graph.json


In [ ]:
from google.colab import files
files.download("final_graph.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import geopandas as gpd

# ---- LOAD GRAPH ----
with open("final_graph_with_rivers.json") as f:
    data = json.load(f)

# ---- LOAD BRIDGES ----
bridges_gdf = gpd.read_file("tokyo_bridges.geojson")

# ---- BUILD EDGE LOOKUP ----
edge_lookup = set()

for _, row in bridges_gdf.iterrows():
    u = row["u"]
    v = row["v"]
    edge_lookup.add((u, v))
    edge_lookup.add((v, u))  # handle both directions

# ---- ADD is_bridge ----
count = 0

for edge in data["edges"]:
    u = edge["from"]
    v = edge["to"]

    if (u, v) in edge_lookup:
        edge["is_bridge"] = True
        count += 1
    else:
        edge["is_bridge"] = False

print("Bridges added:", count)
# ---- BUILD RIVER EDGE LOOKUP ----
river_edges = set()

for _, row in test.iterrows():
    u = row["u"]
    v = row["v"]
    river_edges.add((u, v))
    river_edges.add((v, u))

# ---- APPLY near_river TO JSON ----
river_count = 0

for edge in data["edges"]:
    u = edge["from"]
    v = edge["to"]

    if (u, v) in river_edges:
        edge["near_river"] = True
        river_count += 1
    else:
        edge["near_river"] = False

print("River edges added:", river_count)
# ---- SAVE ----
with open("final_graph_full.json", "w") as f:
    json.dump(data, f)

/usr/local/lib/python3.12/dist-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Bridges added: 3478
River edges added: 2744


In [ ]:
with open("final_graph_full.json") as f:
    data = json.load(f)

print("Bridge edges:",
      sum(1 for e in data["edges"] if e.get("is_bridge")))

Bridge edges: 3478


In [ ]:
import json
from collections import Counter

with open("final_graph_full.json") as f:
    data = json.load(f)

nodes = data["nodes"]
edges = data["edges"]

# ----------------------------
# BASIC COUNTS
# ----------------------------
print("Nodes:", len(nodes))
print("Edges:", len(edges))


# ----------------------------
# NODE PROPERTIES
# ----------------------------
node_keys = set()
for n in nodes:
    node_keys.update(n.keys())

print("\nNode attributes:")
print(node_keys)


# ----------------------------
# EDGE PROPERTIES
# ----------------------------
edge_keys = set()
for e in edges:
    edge_keys.update(e.keys())

print("\nEdge attributes:")
print(edge_keys)


# ----------------------------
# NODE TYPE DISTRIBUTION
# ----------------------------
types = Counter(n.get("type", "unknown") for n in nodes)

print("\nNode type distribution:")
for k, v in types.items():
    print(f"{k}: {v}")


# ----------------------------
# POPULATION STATS
# ----------------------------
pop_values = [n.get("population", 0) for n in nodes if n.get("type") == "building"]

print("\nPopulation stats:")
print("Total buildings:", len(pop_values))
print("Non-zero population:", sum(1 for p in pop_values if p > 0))
print("Max population:", max(pop_values))


# ----------------------------
# EDGE ATTRIBUTE COUNTS
# ----------------------------
near_river = sum(1 for e in edges if e.get("near_river"))
is_bridge = sum(1 for e in edges if e.get("is_bridge"))

print("\nEdge stats:")
print("Near river:", near_river)
print("Bridge:", is_bridge)


# ----------------------------
# COMBO CHECK
# ----------------------------
combo = sum(
    1 for e in edges
    if e.get("near_river") and e.get("is_bridge")
)

print("Bridge + river:", combo)


# ----------------------------
# SAMPLE NODE + EDGE
# ----------------------------
print("\nSample node:")
print(nodes[0])

print("\nSample edge:")
print(edges[0])

Nodes: 77576
Edges: 239021

Node attributes:
{'population', 'type', 'id', 'lat', 'lon'}

Edge attributes:
{'travel_time', 'to', 'length', 'from', 'is_bridge', 'near_river'}

Node type distribution:
building: 10000
road_node: 67094
hospital: 292
fire_station: 190

Population stats:
Total buildings: 10000
Non-zero population: 9859
Max population: 1309

Edge stats:
Near river: 2744
Bridge: 3478
Bridge + river: 1616

Sample node:
{'id': 31236584, 'lat': 35.63493539999999, 'lon': 139.7686834, 'type': 'building', 'population': 0}

Sample edge:
{'from': 31236584, 'to': 31236646, 'length': 971.1623743237018, 'travel_time': 38.84649497294807, 'near_river': False, 'is_bridge': True}


In [ ]:
from google.colab import files
files.download("final_graph_full.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>